# 📘 Async Python Tutorial

A comprehensive, rigorous, self-contained tutorial on async Python.

**What you'll learn:**
- Part 1: Introduction to Async
- Part 2: The Event Loop
- Part 3: `async def` and `await`
- Part 4: Running Async Code from Cursor
- Part 5: Async Code in Gradio Callbacks
- Part 6: Advanced – `asyncio.gather`
- Part 7: Tips, Traps, and Techniques
- Part 8: Exercises

---
## Part 1: Introduction to Async – What, Why, and How?

### 🔄 What Is Async Python?

Async Python is a way to write code that **doesn't block**. Instead of stopping everything while waiting for an operation to finish (like a web request, file read, or sleep), async Python lets other things run during the wait.

This is especially useful for **I/O-bound** operations — things that are slow because of external resources (like the internet or disk), not your CPU.

### 🤹 Async vs Threads vs Multiprocessing

| Feature | `asyncio` (Async) | Threads | Multiprocessing |
|---------|-------------------|---------|------------------|
| Use Case | I/O-bound tasks | I/O-bound (sometimes) | CPU-bound tasks |
| Concurrency | Cooperative | Pre-emptive | True parallelism |
| Overhead | Low | Medium | High |
| Complexity | Medium | Medium | High |
| GIL Aware | Yes | Yes | No |

- **Async** is single-threaded but can handle thousands of concurrent I/O tasks.
- **Threads** allow simultaneous operations but can have race conditions and are limited by the GIL (Global Interpreter Lock).
- **Multiprocessing** bypasses the GIL and runs in multiple processes, good for CPU-heavy tasks.

---
## Part 2: 🧠 The Event Loop

Think of the event loop as a **conductor** managing multiple instruments — tasks. It checks when a task is ready and then runs it, pausing it when it has to wait, and switching to another task.

**Behind the scenes:**
- `say_hello()` is a **coroutine**.
- `asyncio.run()` starts an **event loop**.
- When `await asyncio.sleep(1)` is hit, the event loop **pauses** `say_hello` and can run other coroutines.

In [ ]:
import asyncio

async def say_hello():
    print("Hello")
    await asyncio.sleep(1)  # Pause here, switch to other tasks
    print("World")

# In a notebook, we can use top-level await directly!
await say_hello()

---
## Part 3: ✅ `async def` and `await`

### Basic Usage

**Rules:**
- ✅ Use `async def` to define coroutines
- ✅ Use `await` only inside `async def`
- ❌ Cannot use `await` at top-level in scripts or modules — use `asyncio.run()` instead
- ✅ Return values from async functions with `return`
- ✅ In notebooks, top-level `await` is supported!

In [ ]:
import asyncio

async def greet(name):
    print(f"Hello, {name}")
    await asyncio.sleep(1)
    print(f"Goodbye, {name}")

async def main():
    await greet("Alice")

# In a notebook:
await main()

### 💡 What happens if you forget `await`?

Run the cell below and observe — you'll get a coroutine object instead of running the function!

In [ ]:
# Common mistake: forgetting await
result = greet("Bob")  # No await!
print(f"Result without await: {result}")
print(f"Type: {type(result)}")

# The correct way:
print("\nNow with await:")
await greet("Bob")

---
## Part 4: Running Async Code from Cursor

### ✅ From a Python Module (e.g. `main.py`)

```python
# main.py
import asyncio

async def work():
    await asyncio.sleep(1)
    print("Async done!")

if __name__ == "__main__":
    asyncio.run(work())
```

✅ Just run `python main.py` in the terminal.

### ✅ From a Notebook (in Cursor or Jupyter)

Notebooks support top-level `await`! You **don't** need `asyncio.run()`.

💡 You can also use `nest_asyncio` if you're embedding event loops (e.g., in some servers or LLM apps).

In [ ]:
# This works directly in a notebook — no asyncio.run() needed!
import asyncio

async def hello():
    await asyncio.sleep(1)
    print("Hello from notebook!")

await hello()

---
## Part 5: Async Code in Gradio Callbacks

Gradio lets you use `async def` directly for event handlers like button clicks.

🎉 It just works! Gradio uses `asyncio` under the hood.

> **Note:** Make sure gradio is installed: `uv pip install gradio`

In [ ]:
# Uncomment and run if you have gradio installed
# import gradio as gr
# import asyncio

# async def slow_response(name):
#     await asyncio.sleep(2)
#     return f"Hello, {name}! (after waiting)"

# gr.Interface(fn=slow_response, inputs="text", outputs="text").launch()

---
## Part 6: Advanced Async – `asyncio.gather`

### Run multiple coroutines concurrently

`asyncio.gather()` runs all tasks *in parallel* (as far as I/O waits are concerned). This is where async really shines!

In [ ]:
import asyncio
import time

async def task(name, delay):
    print(f"⏳ {name} starting (will take {delay}s)")
    await asyncio.sleep(delay)
    print(f"✅ {name} done after {delay}s")
    return name

async def main():
    start = time.time()
    results = await asyncio.gather(
        task("A", 1),
        task("B", 2),
        task("C", 3)
    )
    elapsed = time.time() - start
    print(f"\nAll results: {results}")
    print(f"Total time: {elapsed:.1f}s (not 6s!)")

await main()

### 🔍 Compare: Sequential vs Concurrent

Let's see the difference between running tasks one-by-one vs with `gather`:

In [ ]:
import asyncio
import time

async def slow_task(name, delay):
    await asyncio.sleep(delay)
    return f"{name} done"

# Sequential execution
start = time.time()
r1 = await slow_task("A", 1)
r2 = await slow_task("B", 1)
r3 = await slow_task("C", 1)
print(f"Sequential: {time.time() - start:.1f}s → {[r1, r2, r3]}")

# Concurrent execution
start = time.time()
results = await asyncio.gather(
    slow_task("A", 1),
    slow_task("B", 1),
    slow_task("C", 1)
)
print(f"Concurrent: {time.time() - start:.1f}s → {results}")

---
## Part 7: Tips, Traps, and Techniques

### ✅ Tips

- Use `asyncio.gather` to parallelize I/O-bound calls.
- Use `async for` and `async with` for async iterators and context managers.
- Use `anyio` or `trio` for higher-level async if needed.

### ⚠️ Common Traps

| Trap | Fix |
|------|-----|
| Trying to `await` at top-level in a script | Use `asyncio.run()` |
| Mixing `threading` and `asyncio` | Avoid or use with care |
| Forgetting `await` | You'll get a coroutine object and nothing runs |
| Blocking call (e.g., `time.sleep`) | Use `await asyncio.sleep()` instead |

### 🔧 Debugging

Use the following to verify if you're inside a running event loop:

In [ ]:
import asyncio

# Check if an event loop is running
loop = asyncio.get_running_loop()
print(f"Event loop running: {loop.is_running()}")
print(f"Loop type: {type(loop).__name__}")

### 🚫 Trap Demo: Blocking vs Non-blocking sleep

In [ ]:
import asyncio
import time

# BAD: This blocks the entire event loop!
async def bad_example():
    print("Starting bad example (using time.sleep)...")
    time.sleep(2)  # ❌ Blocks everything!
    print("Bad example done")

# GOOD: This lets other tasks run during the wait
async def good_example():
    print("Starting good example (using asyncio.sleep)...")
    await asyncio.sleep(2)  # ✅ Non-blocking
    print("Good example done")

print("=== Bad (blocking) ===")
await bad_example()
print("\n=== Good (non-blocking) ===")
await good_example()

---
## Part 8: 🏋️ Exercises

Try these exercises below! Each exercise has a description and a code cell for your solution.

---

### Exercise 1 – Basic Coroutine

Write a coroutine that:
- Takes a name
- Waits 2 seconds
- Prints `"Hello, [name]!"`

In [ ]:
import asyncio

# Exercise 1: Write your coroutine here
async def greet(name):
    # TODO: implement this
    pass

# Test it:
# await greet("Alice")

<details>
<summary>💡 Click to see solution</summary>

```python
async def greet(name):
    await asyncio.sleep(2)
    print(f"Hello, {name}!")

await greet("Alice")
```
</details>

---
### Exercise 2 – Parallel Coroutines

Write three async functions:
- `fetch_data()` – waits 1 second, returns `"data fetched"`
- `process_data()` – waits 1 second, returns `"data processed"`
- `save_data()` – waits 1 second, returns `"data saved"`

Use `asyncio.gather()` to run all three concurrently.

In [ ]:
import asyncio

# Exercise 2: Write your functions here
async def fetch_data():
    # TODO
    pass

async def process_data():
    # TODO
    pass

async def save_data():
    # TODO
    pass

# Use asyncio.gather to run all three
# results = await asyncio.gather(fetch_data(), process_data(), save_data())
# print(results)

<details>
<summary>💡 Click to see solution</summary>

```python
async def fetch_data():
    await asyncio.sleep(1)
    return "data fetched"

async def process_data():
    await asyncio.sleep(1)
    return "data processed"

async def save_data():
    await asyncio.sleep(1)
    return "data saved"

results = await asyncio.gather(fetch_data(), process_data(), save_data())
print(results)  # Takes ~1s, not 3s!
```
</details>

---
### Exercise 3 – Async Countdown

Write a coroutine that counts down from 5 to 1 with a 1-second wait in between each number.

In [ ]:
import asyncio

# Exercise 3: Async countdown
async def countdown(n):
    # TODO: implement countdown from n to 1
    pass

# Test it:
# await countdown(5)

<details>
<summary>💡 Click to see solution</summary>

```python
async def countdown(n):
    for i in range(n, 0, -1):
        print(i)
        await asyncio.sleep(1)
    print("Go! 🚀")

await countdown(5)
```
</details>

---
### Exercise 4 – Compare Blocking vs Async

Run the code below and compare how long each version takes.
Create a loop of 3 iterations for both blocking and async versions, and time them.

In [ ]:
import asyncio
import time

def blocking():
    time.sleep(1)
    print("Blocking done")

async def non_blocking():
    await asyncio.sleep(1)
    print("Async done")

# Exercise 4: Time 3 iterations of each
# Blocking version:
# start = time.time()
# for _ in range(3):
#     blocking()
# print(f"Blocking total: {time.time() - start:.1f}s")

# Async version (use asyncio.gather!):
# start = time.time()
# await asyncio.gather(*[non_blocking() for _ in range(3)])
# print(f"Async total: {time.time() - start:.1f}s")

<details>
<summary>💡 Click to see solution</summary>

```python
# Blocking version: ~3 seconds
start = time.time()
for _ in range(3):
    blocking()
print(f"Blocking total: {time.time() - start:.1f}s")

# Async version: ~1 second
start = time.time()
await asyncio.gather(*[non_blocking() for _ in range(3)])
print(f"Async total: {time.time() - start:.1f}s")
```

The blocking version takes ~3s (sequential), while the async version takes ~1s (concurrent)!
</details>

---
### Exercise 5 – Build a Gradio Async App

Create a Gradio app that:
- Takes a name
- Waits 2 seconds
- Returns a greeting

Try switching the handler between sync and async and measure the difference in responsiveness.

> Install gradio first: `uv pip install gradio`

In [ ]:
# Exercise 5: Gradio async app
# import gradio as gr
# import asyncio

# Sync version
# def sync_greet(name):
#     import time
#     time.sleep(2)
#     return f"Hello, {name}!"

# Async version
# async def async_greet(name):
#     await asyncio.sleep(2)
#     return f"Hello, {name}!"

# Try both:
# gr.Interface(fn=async_greet, inputs="text", outputs="text").launch()

---
## 🎓 Summary

| Concept | Key Takeaway |
|---------|-------------|
| `async def` | Defines a coroutine |
| `await` | Pauses coroutine, lets other tasks run |
| `asyncio.run()` | Starts event loop (in scripts) |
| Top-level `await` | Works in notebooks |
| `asyncio.gather()` | Run multiple coroutines concurrently |
| `asyncio.sleep()` | Non-blocking sleep |
| `time.sleep()` | ❌ Blocking – avoid in async code |

**Happy async coding! 🚀**